In [133]:
import pandas as pd
import numpy as np
import json
import os
from tqdm import tqdm

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

from openai import OpenAI


In [134]:
# Load API key
with open("key.txt", "r") as f:
    api_key = f.read().strip()

client = OpenAI(api_key=api_key)


In [135]:
import re

# Arabic tashkeel unicode range
TASHKEEL_REGEX = re.compile(r'[\u0617-\u061A\u064B-\u0652]')

def remove_tashkeel(text):
    if not isinstance(text, str):
        return text
    return re.sub(TASHKEEL_REGEX, '', text)


In [136]:
df_pos = pd.read_csv("majaz_dataset.csv")
df_neg = pd.read_csv("non_majaz_random_sample.csv")

df_pos["label"] = 1
df_neg["label"] = 0

TEXT_COL = "البيت_الكامل"

df = pd.concat([df_pos, df_neg], ignore_index=True)
df = df[[TEXT_COL, "label"]].dropna()

# 🔥 REMOVE TASHKEEL
df[TEXT_COL] = df[TEXT_COL].apply(remove_tashkeel)

df.head()


,البيت_الكامل,label
0,مررت على دار الحبيب فحمحمت جوادي وهل تشجو الجي...,1
1,وتسعدني في غمرة بعد غمرة سبوح لها منها عليها ش...,1
2,محرمة أكفال خيلي على القنا محللة لباتها والقلائد,1
3,ولكن إذا لم يحمل القلب كفه على حالة لم يحمل ال...,1
4,له من كريم الطبع في الحرب منتض ومن عادة الإحسا...,1


In [137]:
X_train, X_test, y_train, y_test = train_test_split(
    df[TEXT_COL].values,
    df["label"].values,
    test_size=0.05,
    stratify=df["label"],
    random_state=42
)

len(X_train), len(X_test)


(1514, 80)

In [138]:
SYSTEM_PROMPT = """
أنت خبير متخصص في البلاغة العربية.

التعريف:
المجاز هو استعمال اللفظ في غير معناه الحقيقي الأصلي، مع وجود قرينة تصرف الذهن عن المعنى الحرفي.

الاستبعادات (لا تُعد مجازًا):
- المعاني الحقيقية المباشرة
- التشبيه الصريح باستخدام أدوات مثل: كأن، مثل، كـ
- المبالغة أو التعبير العاطفي دون انتقال دلالي حقيقي
- الألفاظ التي أصبحت مستعملة استعمالًا حقيقيًا شائعًا ما لم يدل السياق على نقل دلالي واضح

قاعدة القرار:
- إذا كان المعنى حرفيًا → الناتج 0
- إذا كان المعنى قائمًا على نقل دلالي (مجاز) → الناتج 1

قواعد الإخراج:
- أخرج رقمًا واحدًا فقط
- 1 → يحتوي على مجاز
- 0 → لا يحتوي على مجاز
- بدون شرح
- بدون أي نص إضافي

"""


In [139]:
def build_prompt(verse):
    return f"""
البيت:
{verse}

السؤال:
هل يحتوي هذا البيت على مجاز وفق البلاغة العربية؟

أجب برقم واحد فقط:
0 أو 1

"""


In [140]:
def build_batch_prompt(verses):
    verses_block = "\n".join(
        [f"{i+1}. {v}" for i, v in enumerate(verses)]
    )

    return f"""
أنت خبير متخصص في البلاغة العربية.

التعريف:
المجاز هو استعمال اللفظ في غير معناه الحقيقي الأصلي، مع وجود قرينة تصرف الذهن عن المعنى الحرفي.

لا يُعد مجازًا:
- المعنى الحرفي
- التشبيه الصريح
- المبالغة اللفظية دون انتقال دلالي
- الألفاظ المستهلكة دلاليًا ما لم يدل السياق على نقل معنوي واضح

المهمة:
لكل بيت من الأبيات التالية، حدّد هل يحتوي على مجاز أم لا.

قواعد الإخراج:
- أرقام فقط
- رقم واحد في كل سطر
- بنفس ترتيب الأبيات
- 1 = يحتوي على مجاز
- 0 = لا يحتوي على مجاز
- بدون شرح أو تعليل

الأبيات:
{verses_block}
"""


In [ ]:
BATCH_SIZE = 20  # 🔥 HARD-CODED

def predict_majaz_batch(verses):
    response = client.responses.create(
        model="gpt-5-mini-2025-08-07",
        input=build_batch_prompt(verses)
    )

    lines = response.output_text.strip().splitlines()

    preds = []
    for line in lines:
        line = line.strip()
        preds.append(1 if line.startswith("1") else 0)

    # safety check
    if len(preds) != len(verses):
        raise ValueError("Batch size mismatch between input and output")

    return preds


In [142]:
y_pred = []

for i in tqdm(range(0, len(X_test), BATCH_SIZE)):
    batch = X_test[i:i+BATCH_SIZE]

    try:
        preds = predict_majaz_batch(batch)
        y_pred.extend(preds)
    except Exception as e:
        print("Batch error:", e)
        y_pred.extend([0] * len(batch))  # safe fallback


100%|██████████| 4/4 [02:34<00:00, 38.52s/it]


In [143]:
acc = accuracy_score(y_test, y_pred)
cm = confusion_matrix(y_test, y_pred)

print("Accuracy:", acc)
print("\nConfusion Matrix:")
print(cm)

print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=["Non-Majaz", "Majaz"]))


Accuracy: 0.575

Confusion Matrix:
[[34  4]
 [30 12]]

Classification Report:
              precision    recall  f1-score   support

   Non-Majaz       0.53      0.89      0.67        38
       Majaz       0.75      0.29      0.41        42

    accuracy                           0.57        80
   macro avg       0.64      0.59      0.54        80
weighted avg       0.65      0.57      0.53        80



In [144]:
# ===============================
# Save GPT predictions to CSV
# ===============================

results_df = pd.DataFrame({
    "verse": X_test,
    "y_true": y_test,
    "y_pred": y_pred
})

OUT_PATH = "gpt_majaz_predictions_test.csv"
results_df.to_csv(OUT_PATH, index=False, encoding="utf-8-sig")

print(f"Saved results to: {OUT_PATH}")
results_df.head()


Saved results to: gpt_majaz_predictions_test.csv


,verse,y_true,y_pred
0,تلوث بعد افتضال البرد مئزرها، لوثا ، على مثل د...,0,0
1,وخيل غزالة تنتابهم تجوب العراق وتحوي النبيطا,1,1
2,في كل رجل ويد زائدة وهو على جرذانه قد شطبا,0,0
3,أإلى بني عبد الكريم تشاوست عيناك ويلك خلف من ت...,0,0
4,دست رسولا أن ائتنا رقدة الـ ـحي إذا ما علمتهم ...,1,1
